In [1]:
# ============================================================
# RETINITIS PIGMENTOSA (RP) DETECTION — PROFESSIONAL SOLUTION
# Model     : ResNet50 Transfer Learning + Fine-Tuning
# Framework : TensorFlow / Keras
# Platform  : Kaggle GPU Notebook (T4 / P100 compatible)
# Classes   : Normal (0) | RP (1)
# ============================================================

# ─────────────────────────────────────────────────────────────
# SECTION 1 — IMPORTS & REPRODUCIBILITY SETUP
# ─────────────────────────────────────────────────────────────
import os
import gc
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')                           # Non-interactive backend for Kaggle
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras

from pathlib import Path
from glob import glob
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, f1_score,
    precision_score, recall_score
)
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (
    GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint,
    ReduceLROnPlateau, CSVLogger
)
from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator, load_img, img_to_array
)

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Output directory ─────────────────────────────────────────
OUT = '/kaggle/working/'
os.makedirs(OUT, exist_ok=True)

# ── GPU configuration ─────────────────────────────────────────
# Memory growth prevents TensorFlow from allocating all GPU memory at once,
# which avoids OOM errors on Kaggle's shared T4/P100 instances.
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs available: {len(gpus)}")
for g in gpus:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except RuntimeError as e:
        print(f"  Memory growth error (GPU already initialised): {e}")

print(f"TensorFlow version : {tf.__version__}")
print(f"Keras version      : {keras.__version__}")


# ─────────────────────────────────────────────────────────────
# SECTION 2 — CENTRALISED CONFIGURATION
# ─────────────────────────────────────────────────────────────
# All hyperparameters and paths live here.
# Change once → takes effect everywhere.

CFG = dict(
    # ── Paths ────────────────────────────────────────────────
    rp_dir   = ('/kaggle/input/datasets/andreivann/'
                'dataset-retinitis-pigmentosa-enhanced/'
                'retinitis-pigmentosa-768-aug'),
    norm_dir = ('/kaggle/input/datasets/falahgatea/'
                'eye-diseases-classification/'
                'eye_diseases_classification/normal'),

    # ── Image ────────────────────────────────────────────────
    img_size = (224, 224),   # ResNet50 native input size
    channels = 3,

    # ── Split ────────────────────────────────────────────────
    val_size  = 0.15,
    test_size = 0.10,        # Held-out; never touches training

    # ── Training ─────────────────────────────────────────────
    batch    = 16,           # Reduced from 32 → safer for Kaggle GPU memory
    epochs_1 = 20,           # Phase-1: head only, frozen backbone
    epochs_2 = 30,           # Phase-2: fine-tune top residual blocks
    lr_1     = 1e-3,
    lr_2     = 1e-5,         # Much lower LR for fine-tuning

    # ── Fine-tune ────────────────────────────────────────────
    # ResNet50 has 175 layers. Unfreezing from index 140 exposes the
    # last ~35 layers (conv5 block + final BN) — the most task-specific.
    unfreeze_from = 140,

    # ── Regularisation ───────────────────────────────────────
    dropout = 0.5,

    # ── Class labels ─────────────────────────────────────────
    # Used by evaluation, Grad-CAM, and plots.
    class_names = ['Normal', 'RP'],
    label_map   = {0: 'Normal', 1: 'RP'},
)

# Convenience aliases used throughout (avoids repetitive CFG lookups)
IMG_H, IMG_W = CFG['img_size']
IMG_SIZE     = IMG_H           # Square images; single value used by load_img
CLASS_NAMES  = CFG['class_names']


# ─────────────────────────────────────────────────────────────
# SECTION 3 — DATASET PREPARATION
# ─────────────────────────────────────────────────────────────

def collect_paths(folder: str, label: int) -> pd.DataFrame:
    """
    Recursively collect all image file paths under `folder`.
    Returns a DataFrame with columns ['filepath', 'label'].
    Label is stored as a string because flow_from_dataframe
    treats y_col values as class names, not integers.
    """
    exts = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff')
    paths = []
    for ext in exts:
        paths.extend(glob(os.path.join(folder, '**', ext), recursive=True))
    return pd.DataFrame({'filepath': paths, 'label': str(label)})


rp_df   = collect_paths(CFG['rp_dir'],   label=1)
norm_df = collect_paths(CFG['norm_dir'], label=0)

print(f"RP images    : {len(rp_df)}")
print(f"Normal images: {len(norm_df)}")

# ── Combine & shuffle ────────────────────────────────────────
full_df = (
    pd.concat([rp_df, norm_df], ignore_index=True)
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)
print(f"Total        : {len(full_df)}")
print(full_df['label'].value_counts())

# ── Stratified three-way split ───────────────────────────────
# Stratify on 'label' so class ratios are preserved in every split.
train_df, temp_df = train_test_split(
    full_df,
    test_size    = CFG['val_size'] + CFG['test_size'],
    stratify     = full_df['label'],
    random_state = SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size    = CFG['test_size'] / (CFG['val_size'] + CFG['test_size']),
    stratify     = temp_df['label'],
    random_state = SEED
)

# Reset indices so iloc-based sampling is safe
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"\nTrain : {len(train_df)}  |  Val : {len(val_df)}  |  Test : {len(test_df)}")
for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"{name} class distribution:\n{df['label'].value_counts()}\n")


# ─────────────────────────────────────────────────────────────
# SECTION 4 — CLASS WEIGHTS
# ─────────────────────────────────────────────────────────────
# FIX: .values returns a numpy array — required by compute_class_weight.
# Without weighting, the majority class dominates loss and the model
# learns to predict it almost always, causing high FP / FN counts.

labels_train = train_df['label'].astype(int).values          # numpy array
cw_values    = compute_class_weight(
    class_weight = 'balanced',
    classes      = np.unique(labels_train),
    y            = labels_train
)
class_weights = {i: float(w) for i, w in enumerate(cw_values)}
print(f"Class weights: {class_weights}")


# ─────────────────────────────────────────────────────────────
# SECTION 5 — DATA GENERATORS
# ─────────────────────────────────────────────────────────────
# KEY RULES:
#   1. preprocess_input scales pixels to [-1, 1] as ResNet50 expects.
#   2. Val/test generators must have shuffle=False so that
#      generator.labels aligns with model.predict() output.
#   3. Augmentation on training only — never on val/test.

train_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    horizontal_flip        = True,
    vertical_flip          = False,      # Anatomically unnatural for fundus
    rotation_range         = 15,
    zoom_range             = 0.15,
    shear_range            = 0.10,
    width_shift_range      = 0.10,
    height_shift_range     = 0.10,
    brightness_range       = [0.85, 1.15],
    fill_mode              = 'nearest'
)

eval_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input   # Rescale only; no augmentation
)


def make_generator(datagen, df, shuffle: bool, batch_size: int = CFG['batch']):
    """Create a flow_from_dataframe generator with consistent settings."""
    return datagen.flow_from_dataframe(
        dataframe   = df,
        x_col       = 'filepath',
        y_col       = 'label',
        target_size = CFG['img_size'],
        color_mode  = 'rgb',
        class_mode  = 'binary',
        batch_size  = batch_size,
        shuffle     = shuffle,
        seed        = SEED
    )


train_gen = make_generator(train_datagen, train_df, shuffle=True)
val_gen   = make_generator(eval_datagen,  val_df,   shuffle=False)  # CRITICAL: False
test_gen  = make_generator(eval_datagen,  test_df,  shuffle=False)  # CRITICAL: False

print(f"\nClass indices : {train_gen.class_indices}")
# Expected: {'0': 0, '1': 1}  →  Normal=0, RP=1


# ─────────────────────────────────────────────────────────────
# SECTION 6 — MODEL ARCHITECTURE
# ─────────────────────────────────────────────────────────────
# Strategy:
#   Phase-1 — Freeze entire ResNet50; train only the custom head.
#             Avoids destroying pre-trained ImageNet features at high LR.
#   Phase-2 — Unfreeze top N layers; fine-tune at very low LR.
#             Adapts conv5 features from generic textures to retinal patterns.
#
# NAMING CONTRACT (must match compute_gradcam head layer access):
#   GAP layer         → name='gap'
#   BatchNorm         → name='bn_head'
#   Dense 256         → name='fc1'
#   Dropout 0.5       → name='drop1'
#   Dense 128         → name='fc2'
#   Dropout 0.3       → name='drop2'
#   Output sigmoid    → name='predictions'

def build_model(trainable_base: bool = False) -> tuple:
    """
    Build ResNet50 transfer-learning model.

    Returns
    -------
    model      : Full Keras Model (input → predictions).
    base_model : The ResNet50 sub-model (used for layer unfreezing in Phase-2).
    """
    base = ResNet50(
        weights     = 'imagenet',
        include_top = False,
        input_shape = (IMG_H, IMG_W, CFG['channels'])
    )
    base.trainable = trainable_base

    # Custom classification head
    x   = base.output
    x   = GlobalAveragePooling2D(name='gap')(x)
    x   = BatchNormalization(name='bn_head')(x)
    x   = Dense(256, activation='relu', name='fc1')(x)
    x   = Dropout(CFG['dropout'],  name='drop1')(x)
    x   = Dense(128, activation='relu', name='fc2')(x)
    x   = Dropout(0.3,             name='drop2')(x)
    out = Dense(1, activation='sigmoid', name='predictions')(x)

    model = Model(inputs=base.input, outputs=out)
    return model, base


def compile_model(model: Model, lr: float) -> Model:
    """Compile model with Adam + binary cross-entropy + clinical metrics."""
    model.compile(
        optimizer = Adam(learning_rate=lr),
        loss      = 'binary_crossentropy',
        metrics   = [
            'accuracy',
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ]
    )
    return model


model, base_model = build_model(trainable_base=False)
model = compile_model(model, lr=CFG['lr_1'])

model.summary()
print(f"\nTotal layers in ResNet50 base : {len(base_model.layers)}")
print(f"Top-level model layers        : {[l.name for l in model.layers]}")


# ─────────────────────────────────────────────────────────────
# SECTION 7 — CALLBACKS
# ─────────────────────────────────────────────────────────────

def get_callbacks(phase: int) -> list:
    """Return a standard callback set for the given training phase."""
    ckpt_path = os.path.join(OUT, f'best_model_phase{phase}.keras')
    return [
        EarlyStopping(
            monitor              = 'val_auc',
            patience             = 7,
            mode                 = 'max',
            restore_best_weights = True,
            verbose              = 1
        ),
        ModelCheckpoint(
            filepath       = ckpt_path,
            monitor        = 'val_auc',
            save_best_only = True,
            mode           = 'max',
            verbose        = 1
        ),
        ReduceLROnPlateau(
            monitor  = 'val_loss',
            factor   = 0.5,
            patience = 4,
            min_lr   = 1e-7,
            verbose  = 1
        ),
        # CSVLogger ensures training history survives a kernel restart.
        # Graphs can be regenerated from CSV without re-training.
        CSVLogger(os.path.join(OUT, f'training_log_phase{phase}.csv'))
    ]


# ─────────────────────────────────────────────────────────────
# SECTION 8 — PHASE-1 TRAINING  (frozen backbone)
# ─────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("PHASE 1 — Training head with frozen ResNet50 backbone")
print("=" * 60)

history1 = model.fit(
    train_gen,
    epochs          = CFG['epochs_1'],
    validation_data = val_gen,
    class_weight    = class_weights,
    callbacks       = get_callbacks(phase=1),
    verbose         = 1
)


# ─────────────────────────────────────────────────────────────
# SECTION 9 — PHASE-2 FINE-TUNING  (top ResNet50 layers)
# ─────────────────────────────────────────────────────────────
# We selectively unfreeze only the top residual block (conv5) of ResNet50.
# This lets the model adapt high-level feature detectors to retinal textures
# (pigment spicules, vessel attenuation) without catastrophic forgetting.

print("\n" + "=" * 60)
print("PHASE 2 — Fine-tuning top ResNet50 layers")
print("=" * 60)

# Freeze everything first, then unfreeze from the chosen index onward
for layer in base_model.layers:
    layer.trainable = False
for layer in base_model.layers[CFG['unfreeze_from']:]:
    layer.trainable = True

trainable_count = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f"Trainable parameters after unfreeze: {trainable_count:,}")

# Recompile is mandatory after changing trainability
model = compile_model(model, lr=CFG['lr_2'])

history2 = model.fit(
    train_gen,
    epochs          = CFG['epochs_2'],
    validation_data = val_gen,
    class_weight    = class_weights,
    callbacks       = get_callbacks(phase=2),
    verbose         = 1
)


# ─────────────────────────────────────────────────────────────
# SECTION 10 — TRAINING CURVES
# ─────────────────────────────────────────────────────────────

def merge_histories(h1, h2) -> dict:
    """
    Concatenate Phase-1 and Phase-2 Keras History objects into one dict.
    Keys present in h1 but absent in h2 are extended with empty lists
    to prevent KeyError.
    """
    merged = {}
    all_keys = set(h1.history.keys()) | set(h2.history.keys())
    for k in all_keys:
        merged[k] = h1.history.get(k, []) + h2.history.get(k, [])
    return merged


hist           = merge_histories(history1, history2)
p1_epoch_count = len(history1.history['loss'])   # Where vertical divider goes


def plot_curves(hist: dict, p1_end: int, save_path: str) -> None:
    """Plot accuracy, loss, and AUC curves for both training phases."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    pairs = [
        ('accuracy', 'val_accuracy', 'Accuracy'),
        ('loss',     'val_loss',     'Loss'),
        ('auc',      'val_auc',      'ROC-AUC'),
    ]
    for ax, (train_key, val_key, title) in zip(axes, pairs):
        if train_key in hist:
            ax.plot(hist[train_key], label=f'Train {title}', linewidth=2)
        if val_key in hist:
            ax.plot(hist[val_key],   label=f'Val {title}',   linewidth=2)
        ax.axvline(p1_end, color='gray', linestyle='--', linewidth=1,
                   label='Fine-tune start')
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('Epoch', fontsize=11)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.suptitle('Retinitis Pigmentosa Detection — Training Curves',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_path}")


plot_curves(hist, p1_epoch_count,
            save_path=os.path.join(OUT, 'training_curves.png'))


# ─────────────────────────────────────────────────────────────
# SECTION 11 — LOAD BEST MODEL & PREDICT ON TEST SET
# ─────────────────────────────────────────────────────────────
# We load the checkpoint saved by ModelCheckpoint (best val_auc epoch)
# rather than using the last epoch's weights.
# The test set has never influenced training — it gives an unbiased estimate.

best_ckpt = os.path.join(OUT, 'best_model_phase2.keras')
if not os.path.exists(best_ckpt):
    best_ckpt = os.path.join(OUT, 'best_model_phase1.keras')

model = tf.keras.models.load_model(best_ckpt)
print(f"\nLoaded checkpoint: {best_ckpt}")
print(f"Model layer names: {[l.name for l in model.layers]}")

# ── Predict ──────────────────────────────────────────────────
# FIX: reset() before predict() ensures the generator restarts from
# index 0, so predictions[i] corresponds to test_gen.labels[i].
test_gen.reset()
y_pred_prob = model.predict(test_gen, verbose=1).flatten()   # shape: (N,)
y_true      = test_gen.labels                                # numpy int array

assert len(y_true) == len(y_pred_prob), (
    f"Label/prediction count mismatch: {len(y_true)} vs {len(y_pred_prob)}"
)

# ── Optimal threshold via Youden's J statistic ───────────────
# The default 0.5 threshold is arbitrary. For imbalanced medical screening,
# Youden's J (TPR - FPR) finds the cut-point that best balances
# sensitivity and specificity.
fpr, tpr, thresholds = roc_curve(y_true, y_pred_prob)
youden_idx        = int(np.argmax(tpr - fpr))
optimal_threshold = float(thresholds[youden_idx])
print(f"Optimal threshold (Youden's J): {optimal_threshold:.4f}")

y_pred = (y_pred_prob >= optimal_threshold).astype(int)


# ─────────────────────────────────────────────────────────────
# SECTION 12 — CONFUSION MATRIX
# ─────────────────────────────────────────────────────────────

cm             = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"\nConfusion Matrix:")
print(f"  TN={tn}  FP={fp}")
print(f"  FN={fn}  TP={tp}")

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Normal (Pred)', 'RP (Pred)'],
    yticklabels=['Normal (True)', 'RP (True)'],
    ax=ax, linewidths=0.5, annot_kws={'size': 14, 'weight': 'bold'}
)
ax.set_title('Confusion Matrix — Test Set', fontsize=13, fontweight='bold')
ax.set_ylabel('True Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'confusion_matrix.png'), dpi=150)
plt.close()
print("Saved: confusion_matrix.png")


# ─────────────────────────────────────────────────────────────
# SECTION 13 — COMPREHENSIVE METRICS
# ─────────────────────────────────────────────────────────────

sensitivity = tp / (tp + fn + 1e-9)    # Recall for RP  (True Positive Rate)
specificity = tn / (tn + fp + 1e-9)    # Recall for Normal (True Negative Rate)
ppv         = tp / (tp + fp + 1e-9)    # Precision / Positive Predictive Value
npv         = tn / (tn + fn + 1e-9)    # Negative Predictive Value
f1          = f1_score(y_true, y_pred)
roc_auc     = roc_auc_score(y_true, y_pred_prob)
accuracy    = (tp + tn) / (tp + tn + fp + fn)

print("\n" + "=" * 50)
print("EVALUATION METRICS — TEST SET")
print("=" * 50)
print(f"Accuracy    : {accuracy:.4f}")
print(f"Sensitivity : {sensitivity:.4f}  (RP Recall — True RP caught)")
print(f"Specificity : {specificity:.4f}  (Normal Recall — correctly rejected)")
print(f"Precision   : {ppv:.4f}  (PPV — of RP predictions, how many correct)")
print(f"NPV         : {npv:.4f}  (of Normal predictions, how many correct)")
print(f"F1-Score    : {f1:.4f}")
print(f"ROC-AUC     : {roc_auc:.4f}")
print("=" * 50)
print("\nFull Classification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Save metrics to CSV
metrics_df = pd.DataFrame([{
    'Accuracy': accuracy, 'Sensitivity': sensitivity,
    'Specificity': specificity, 'Precision_PPV': ppv, 'NPV': npv,
    'F1_Score': f1, 'ROC_AUC': roc_auc,
    'Threshold': optimal_threshold,
    'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)
}])
metrics_df.to_csv(os.path.join(OUT, 'metrics.csv'), index=False)
print("Saved: metrics.csv")


# ─────────────────────────────────────────────────────────────
# SECTION 14 — ROC CURVE
# ─────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='darkorange', lw=2,
        label=f'ROC Curve (AUC = {roc_auc:.4f})')
ax.scatter(fpr[youden_idx], tpr[youden_idx], s=120, color='red', zorder=5,
           label=f'Optimal threshold = {optimal_threshold:.3f}')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.set_xlabel('False Positive Rate (1 − Specificity)', fontsize=12)
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
ax.set_title('ROC Curve — Retinitis Pigmentosa Detection',
             fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'roc_curve.png'), dpi=150)
plt.close()
print("Saved: roc_curve.png")


# ─────────────────────────────────────────────────────────────
# SECTION 15 — GRAD-CAM IMPLEMENTATION  (fully corrected)
# ─────────────────────────────────────────────────────────────
#
# ARCHITECTURE NOTE
# ─────────────────
# build_model() uses ResNet50 as a *sub-model* object named 'resnet50'
# embedded inside the top-level Model.  The layer hierarchy is:
#
#   top-level model
#     ├── <resnet50>          ← keras sub-model, name='resnet50'
#     │    └── conv5_block3_out  ← INSIDE the sub-model
#     ├── gap
#     ├── bn_head
#     ├── fc1                 ← CORRECT name from build_model()
#     ├── drop1               ← CORRECT name from build_model()
#     ├── fc2                 ← CORRECT name from build_model()
#     ├── drop2               ← CORRECT name from build_model()
#     └── predictions
#
# FIXES APPLIED
# ─────────────
# 1. Layer name mismatch:
#      WRONG  → model.get_layer('dense_1') / 'dropout_1' / 'dense_2' / 'dropout_2'
#      FIXED  → model.get_layer('fc1') / 'drop1' / 'fc2' / 'drop2'
#    These names are defined in build_model() and must be used consistently.
#
# 2. Nested sub-model access:
#      WRONG  → model.get_layer('conv5_block3_out')  (not a top-level layer)
#      FIXED  → model.get_layer('resnet50').get_layer('conv5_block3_out')
#
# 3. GradientTape scope:
#    conv_outputs must be watched *inside* the tape context.
#    We build a backbone-only sub-model so the tape can track the tensor.
#
# 4. Tensor → numpy conversion:
#    Done exactly once per tensor (.numpy() on a tf.Tensor only).
#    Subsequent operations use the numpy arrays directly.
#
# 5. Retinal masking applied before compute_gradcam() to suppress
#    uninformative black-border activations.

def apply_retinal_mask(image_array: np.ndarray) -> np.ndarray:
    """
    Zero out pixels outside a circular region centred on the image.

    Retinal fundus photographs have a circular field of view surrounded
    by black borders. Without masking, Grad-CAM can attribute importance
    to these uninformative borders rather than pathological features.

    Parameters
    ----------
    image_array : np.ndarray, shape (H, W, 3), values in [0, 1].

    Returns
    -------
    Masked image array of the same shape.
    """
    h, w = image_array.shape[:2]
    cx, cy = w // 2, h // 2
    radius = min(cx, cy) - 5

    Y, X  = np.ogrid[:h, :w]
    dist  = np.sqrt((X - cx) ** 2 + (Y - cy) ** 2)
    mask  = (dist <= radius).astype(np.float32)[:, :, np.newaxis]  # (H,W,1)

    return image_array * mask


def _build_gradcam_submodel(model: Model,
                            last_conv_layer_name: str) -> Model:
    """
    Build a sub-model that outputs both the target conv feature maps
    and the ResNet50 backbone output.

    This sub-model covers only the ResNet50 portion so that
    tf.GradientTape can track gradients w.r.t. the conv outputs.

    Parameters
    ----------
    model               : Top-level Keras model.
    last_conv_layer_name: Name of the target conv layer inside ResNet50.

    Returns
    -------
    grad_model : Model(backbone_input → [conv_output, backbone_output])
    """
    # Step 1: Navigate into the nested ResNet50 sub-model
    try:
        backbone = model.get_layer('resnet50')
    except ValueError:
        available = [l.name for l in model.layers]
        raise ValueError(
            f"Sub-model 'resnet50' not found in top-level model. "
            f"Available top-level layers: {available}"
        )

    # Step 2: Find the target conv layer inside the backbone
    try:
        target_layer = backbone.get_layer(last_conv_layer_name)
    except ValueError:
        conv_names = [l.name for l in backbone.layers
                      if 'conv' in l.name.lower()]
        raise ValueError(
            f"Layer '{last_conv_layer_name}' not found inside ResNet50. "
            f"Available conv layers (last 5): {conv_names[-5:]}"
        )

    # Step 3: Build sub-model over backbone only
    grad_model = Model(
        inputs  = backbone.inputs,
        outputs = [target_layer.output, backbone.output],
        name    = 'gradcam_submodel'
    )
    return grad_model


def compute_gradcam(model: Model,
                    img_array: np.ndarray,
                    last_conv_layer_name: str = 'conv5_block3_out'
                    ) -> tuple:
    """
    Compute a Grad-CAM saliency heatmap for a single retinal image.

    Algorithm
    ---------
    1. Run the backbone sub-model with GradientTape to get conv feature
       maps and their gradients w.r.t. the predicted RP probability.
    2. Global-average-pool the gradients → per-channel importance weights.
    3. Weight each feature map by its importance; average → raw heatmap.
    4. Apply ReLU (only positive contributions) and normalise to [0, 1].
    5. Manually apply the classification head to get the final probability.

    Parameters
    ----------
    model               : Top-level Keras model (contains 'resnet50' sub-model).
    img_array           : Float32 numpy array, shape (H, W, 3), values in [0, 1].
                          Apply preprocess_input BEFORE calling this function if
                          the model was trained with preprocess_input.
    last_conv_layer_name: Name of the target conv layer inside ResNet50
                          (default: 'conv5_block3_out').

    Returns
    -------
    heatmap   : 2-D float32 numpy array, shape (h, w), values in [0, 1].
    pred_prob : Scalar float — sigmoid RP probability for this image.
    """
    grad_model = _build_gradcam_submodel(model, last_conv_layer_name)

    # Add batch dimension; keep as float32 tensor for the tape
    img_tensor = tf.cast(tf.expand_dims(img_array, axis=0), dtype=tf.float32)

    with tf.GradientTape() as tape:
        # Watch the input tensor explicitly so we get layer-output gradients
        tape.watch(img_tensor)

        # Forward pass through backbone sub-model
        conv_outputs_t, backbone_out_t = grad_model(
            img_tensor, training=False
        )

        # ── Classification head — FIXED layer names ──────────────────────
        # Uses the exact names defined in build_model():
        #   fc1, drop1, fc2, drop2, predictions
        # NOT the Keras auto-names dense_1 / dropout_1 / dense_2 / dropout_2
        x = model.get_layer('gap')(backbone_out_t)
        x = model.get_layer('bn_head')(x,    training=False)
        x = model.get_layer('fc1')(x)
        x = model.get_layer('drop1')(x,      training=False)
        x = model.get_layer('fc2')(x)
        x = model.get_layer('drop2')(x,      training=False)
        predictions_t = model.get_layer('predictions')(x)

        # Scalar loss: RP sigmoid probability
        loss = predictions_t[:, 0]

    # Gradients of RP probability w.r.t. conv feature maps
    grads_t = tape.gradient(loss, conv_outputs_t)   # (1, H, W, C)

    # Global average pool → per-channel importance weights: shape (C,)
    pooled_grads = tf.reduce_mean(grads_t, axis=(0, 1, 2))

    # ── Convert tensors to numpy ONCE ─────────────────────────────────────
    conv_np   = conv_outputs_t[0].numpy()   # (H_feat, W_feat, C)
    grads_np  = pooled_grads.numpy()        # (C,)

    # Weight feature maps by gradient importance
    weighted  = conv_np * grads_np[np.newaxis, np.newaxis, :]   # (H, W, C)

    # Collapse channels → 2-D heatmap
    heatmap = np.mean(weighted, axis=-1)    # (H_feat, W_feat)

    # ReLU: retain only features that increase RP probability
    heatmap = np.maximum(heatmap, 0.0)

    # Normalise to [0, 1]; guard against all-zero heatmap
    h_max = heatmap.max()
    if h_max > 1e-8:
        heatmap = heatmap / h_max

    # Safe scalar extraction of prediction probability
    pred_prob = float(predictions_t[0, 0].numpy())

    # Explicitly delete large tensors to free GPU memory
    del grad_model, conv_outputs_t, backbone_out_t, grads_t, img_tensor
    gc.collect()

    return heatmap.astype(np.float32), pred_prob


def _resize_heatmap(heatmap: np.ndarray, target_h: int, target_w: int) -> np.ndarray:
    """
    Resize a 2-D heatmap to (target_h, target_w).

    Uses skimage if available; falls back to cv2 bicubic interpolation.
    Both return plain numpy arrays — no .numpy() needed.
    """
    try:
        from skimage.transform import resize as sk_resize
        return sk_resize(heatmap, (target_h, target_w),
                         anti_aliasing=True).astype(np.float32)
    except ImportError:
        import cv2
        return cv2.resize(heatmap, (target_w, target_h),
                          interpolation=cv2.INTER_CUBIC).astype(np.float32)


def visualize_gradcam_batch(model: Model,
                            df_source: pd.DataFrame,
                            n_samples: int = 8,
                            last_conv_layer: str = 'conv5_block3_out',
                            save_path: str = None) -> None:
    """
    Visualise Grad-CAM saliency for a random batch of retinal images.

    Each row shows three panels:
        [Original Image]  |  [Grad-CAM Heatmap]  |  [Overlay]

    Parameters
    ----------
    model           : Trained top-level Keras model.
    df_source       : DataFrame with columns ['filepath', 'label'].
    n_samples       : Number of images to visualise (default 8).
    last_conv_layer : Conv layer name inside ResNet50.
    save_path       : Full path to save the figure (PNG).
    """
    n_samples = min(n_samples, len(df_source))

    # Reproducible random selection
    rng     = np.random.default_rng(seed=SEED)
    indices = rng.choice(len(df_source), size=n_samples, replace=False)

    all_paths  = df_source['filepath'].tolist()
    all_labels = df_source['label'].astype(int).tolist()

    # ── Figure layout ─────────────────────────────────────────
    n_rows, n_cols = n_samples, 3
    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(13, 4.2 * n_rows),
                              squeeze=False)   # FIX: squeeze=False → always 2-D axes array
    fig.suptitle(
        'Grad-CAM Explainability — Retinitis Pigmentosa vs Normal\n'
        'Red = High Model Attention   |   Blue = Low Model Attention',
        fontsize=13, fontweight='bold', y=1.01
    )

    # Column header titles on the first row
    for col_idx, title in enumerate(['Original Image', 'Grad-CAM Heatmap', 'Overlay']):
        axes[0, col_idx].set_title(title, fontsize=11, fontweight='bold', pad=8)

    # ── Per-image loop ────────────────────────────────────────
    for row_idx, img_idx in enumerate(indices):
        img_path   = all_paths[img_idx]
        true_label = all_labels[img_idx]
        true_name  = CLASS_NAMES[true_label]

        # Load original image (not preprocessed) for display
        img_raw   = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
        img_disp  = img_to_array(img_raw) / 255.0           # [0,1] for display

        # Apply ResNet50 preprocessing for Grad-CAM computation
        img_for_model = preprocess_input(
            img_to_array(img_raw).copy()                    # preprocess_input modifies in-place
        )

        # Apply retinal circular mask to suppress border activations
        img_masked = apply_retinal_mask(img_for_model)

        # Compute Grad-CAM; catch per-sample failures gracefully
        try:
            heatmap, pred_prob = compute_gradcam(
                model, img_masked, last_conv_layer
            )
        except Exception as exc:
            print(f"  ⚠️  Grad-CAM failed for sample index {img_idx}: {exc}")
            for col_idx in range(n_cols):
                axes[row_idx, col_idx].axis('off')
                axes[row_idx, col_idx].set_facecolor('#f0f0f0')
            continue

        # Prediction labels and display colours
        pred_class = int(pred_prob >= optimal_threshold)
        pred_name  = CLASS_NAMES[pred_class]
        confidence = pred_prob if pred_class == 1 else (1.0 - pred_prob)
        is_correct = (pred_class == true_label)
        result_sym = '✓' if is_correct else '✗'
        label_col  = '#1b5e20' if is_correct else '#b71c1c'

        # Resize heatmap from backbone spatial dims (7×7) to image size
        heatmap_full = _resize_heatmap(heatmap, IMG_SIZE, IMG_SIZE)

        # Colour-map the heatmap and blend with original image
        heatmap_rgb = plt.cm.jet(heatmap_full)[:, :, :3]             # (H,W,3) float
        overlay     = np.clip(0.5 * img_disp + 0.5 * heatmap_rgb,
                              0.0, 1.0).astype(np.float32)

        # ── Column 0: Original image ──────────────────────────
        axes[row_idx, 0].imshow(img_disp)
        axes[row_idx, 0].set_ylabel(
            f'True: {true_name}', fontsize=9,
            rotation=90, labelpad=4, va='center'
        )

        # ── Column 1: Heatmap ─────────────────────────────────
        axes[row_idx, 1].imshow(heatmap_full, cmap='jet', vmin=0, vmax=1)
        axes[row_idx, 1].set_xlabel(
            f'Confidence: {confidence:.1%}', fontsize=9
        )

        # ── Column 2: Overlay ─────────────────────────────────
        axes[row_idx, 2].imshow(overlay)
        axes[row_idx, 2].set_xlabel(
            f'Pred: {pred_name}  {result_sym}',
            fontsize=9, color=label_col, fontweight='bold'
        )

        # Remove tick marks from all three panels in this row
        for col_idx in range(n_cols):
            axes[row_idx, col_idx].set_xticks([])
            axes[row_idx, col_idx].set_yticks([])

    # ── Shared activation-intensity colour-bar ────────────────
    cbar_ax = fig.add_axes([0.36, -0.01, 0.28, 0.012])
    sm = plt.cm.ScalarMappable(
        cmap='jet', norm=plt.Normalize(vmin=0, vmax=1)
    )
    sm.set_array([])
    fig.colorbar(sm, cax=cbar_ax, orientation='horizontal',
                 label='Activation Intensity')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        print(f"✅ Grad-CAM visualization saved → {save_path}")

    plt.close()
    gc.collect()
    print(f"✅ Grad-CAM complete — {n_samples} samples processed")


# ── Run Grad-CAM on a random sample from the test set ────────
print("\n" + "=" * 60)
print("GENERATING GRAD-CAM VISUALIZATIONS")
print("=" * 60)

visualize_gradcam_batch(
    model           = model,
    df_source       = test_df,
    n_samples       = 8,
    last_conv_layer = 'conv5_block3_out',
    save_path       = os.path.join(OUT, 'gradcam_visualization.png')
)


# ─────────────────────────────────────────────────────────────
# SECTION 16 — PER-CLASS ACCURACY BAR CHART
# ─────────────────────────────────────────────────────────────

class_accuracy = {'Normal': float(specificity), 'RP': float(sensitivity)}

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(class_accuracy.keys(), class_accuracy.values(),
              color=['steelblue', 'tomato'], edgecolor='black', width=0.5)
ax.set_ylim([0, 1.15])
ax.set_ylabel('Accuracy (Recall per class)', fontsize=12)
ax.set_title('Per-Class Detection Accuracy', fontsize=13, fontweight='bold')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='Chance level')
ax.legend(fontsize=10)
for bar, val in zip(bars, class_accuracy.values()):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'per_class_accuracy.png'), dpi=150)
plt.close()
print("Saved: per_class_accuracy.png")


# ─────────────────────────────────────────────────────────────
# SECTION 17 — SAVE FINAL MODEL
# ─────────────────────────────────────────────────────────────

# .keras format (recommended for TF ≥ 2.12)
model.save(os.path.join(OUT, 'rp_detection_final.keras'))
print("Saved: rp_detection_final.keras")

# .h5 for broad compatibility
model.save(os.path.join(OUT, 'rp_detection_final.h5'))
print("Saved: rp_detection_final.h5")

# Architecture JSON for documentation / inference server loading
with open(os.path.join(OUT, 'model_architecture.json'), 'w') as fh:
    fh.write(model.to_json())
print("Saved: model_architecture.json")


# ─────────────────────────────────────────────────────────────
# SECTION 18 — FINAL SUMMARY REPORT
# ─────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("FINAL SUMMARY REPORT")
print("=" * 60)
print(f"Model         : ResNet50 + Custom Head (Two-Phase Transfer Learning)")
print(f"Total epochs  : {len(hist['loss'])}")
print(f"Optimal τ     : {optimal_threshold:.4f}  (Youden's J)")
print(f"Accuracy      : {accuracy:.4f}")
print(f"Sensitivity   : {sensitivity:.4f}  ← RP detection rate")
print(f"Specificity   : {specificity:.4f}  ← Normal detection rate")
print(f"F1-Score      : {f1:.4f}")
print(f"ROC-AUC       : {roc_auc:.4f}")
print(f"TN | FP       : {int(tn)} | {int(fp)}")
print(f"FN | TP       : {int(fn)} | {int(tp)}")
print("=" * 60)

print("\nFiles saved to /kaggle/working/:")
for fname in sorted(os.listdir(OUT)):
    fpath = os.path.join(OUT, fname)
    if os.path.isfile(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f"  {fname:<50s} {size_kb:>8.1f} KB")

print("\nDONE.")

2026-05-19 19:48:38.140749: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779220118.347257      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779220118.405415      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779220118.878180      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779220118.878224      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779220118.878227      23 computation_placer.cc:177] computation placer alr

GPUs available: 1
TensorFlow version : 2.19.0
Keras version      : 3.10.0
RP images    : 8028
Normal images: 1074
Total        : 9102
label
1    8028
0    1074
Name: count, dtype: int64

Train : 6826  |  Val : 1365  |  Test : 911
Train class distribution:
label
1    6021
0     805
Name: count, dtype: int64

Val class distribution:
label
1    1204
0     161
Name: count, dtype: int64

Test class distribution:
label
1    803
0    108
Name: count, dtype: int64

Class weights: {0: 4.239751552795031, 1: 0.5668493605713336}
Found 6826 validated image filenames belonging to 2 classes.
Found 1365 validated image filenames belonging to 2 classes.
Found 911 validated image filenames belonging to 2 classes.

Class indices : {'0': 0, '1': 1}


I0000 00:00:1779220168.412574      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


Model: "functional"

...

I0000 00:00:1779220185.616144      67 service.cc:152] XLA service 0x79afdc003aa0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779220185.616175      67 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1779220187.639572      67 cuda_dnn.cc:529] Loaded cuDNN version 91002


  1/427 ━━━━━━━━━━━━━━━━━━━━ 1:38:20 14s/step - accuracy: 0.5625 - auc: 1.0000 - loss: 0.4069 - precision: 1.0000 - recall: 0.5000

I0000 00:00:1779220192.211954      67 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


427/427 ━━━━━━━━━━━━━━━━━━━━ 0s 305ms/step - accuracy: 0.9293 - auc: 0.9670 - loss: 0.2535 - precision: 0.9864 - recall: 0.9329
Epoch 1: val_auc improved from -inf to 0.99603, saving model to /kaggle/working/best_model_phase1.keras
427/427 ━━━━━━━━━━━━━━━━━━━━ 164s 353ms/step - accuracy: 0.9293 - auc: 0.9670 - loss: 0.2533 - precision: 0.9864 - recall: 0.9329 - val_accuracy: 0.9516 - val_auc: 0.9960 - val_loss: 0.1453 - val_precision: 1.0000 - val_recall: 0.9452 - learning_rate: 0.0010
Epoch 2/20
427/427 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step - accuracy: 0.9679 - auc: 0.9914 - loss: 0.1165 - precision: 0.9958 - recall: 0.9675
Epoch 2: val_auc improved from 0.99603 to 0.99831, saving model to /kaggle/working/best_model_phase1.keras
427/427 ━━━━━━━━━━━━━━━━━━━━ 111s 259ms/step - accuracy: 0.9679 - auc: 0.9914 - loss: 0.1165 - precision: 0.9958 - recall: 0.9675 - val_accuracy: 0.9538 - val_auc: 0.9983 - val_loss: 0.1167 - val_precision: 1.0000 - val_recall: 0.9477 - learning_rate: 0.0010
Epoc

Saved: rp_detection_final.keras
Saved: rp_detection_final.h5
Saved: model_architecture.json

FINAL SUMMARY REPORT
Model         : ResNet50 + Custom Head (Two-Phase Transfer Learning)
Total epochs  : 24
Optimal τ     : 0.3176  (Youden's J)
Accuracy      : 1.0000
Sensitivity   : 1.0000  ← RP detection rate
Specificity   : 1.0000  ← Normal detection rate
F1-Score      : 1.0000
ROC-AUC       : 1.0000
TN | FP       : 108 | 0
FN | TP       : 0 | 803

Files saved to /kaggle/working/:
  __notebook__.ipynb                                   6457.1 KB
  best_model_phase1.keras                             99395.3 KB
  best_model_phase2.keras                            216441.3 KB
  confusion_matrix.png                                   43.9 KB
  gradcam_visualization.png                              66.1 KB
  metrics.csv                                             0.2 KB
  model_architecture.json                               164.3 KB
  per_class_accuracy.png                                 34.4 K